# Step 4: Create Knowledge Graph Triples

This notebook creates knowledge graph triples from preprocessed data:
1. USDA triples: Food → Ingredients, Food → Brands
2. PubMed triples: Ingredient → Disease (with evidence)
3. Save as CSV chunks + GraphML

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import networkx as nx
from datetime import datetime

In [2]:
# Define paths
USDA_DATA_PATH = '../data/pre-processed/usda/'
PUBMED_DATA_PATH = '../data/pre-processed/pubmed/'
KG_OUTPUT_PATH = '../data/pre-processed/kg/'

# Ensure output directory exists
os.makedirs(KG_OUTPUT_PATH, exist_ok=True)
print(f"Output directory: {KG_OUTPUT_PATH}")

Output directory: ../data/pre-processed/kg/


## Part 1: USDA Knowledge Graph Triples

**Note:** USDA triples may already exist in `kg/triples_part_*.csv`. Set `REGENERATE_USDA = True` to regenerate.

In [3]:
# Configuration
REGENERATE_USDA = False  # Set to True to regenerate USDA triples

# Check if USDA triples already exist
existing_usda_triples = [f for f in os.listdir(KG_OUTPUT_PATH) if f.startswith('triples_part_') and f.endswith('.csv')]
print(f"Existing USDA triple files: {len(existing_usda_triples)}")
for f in existing_usda_triples:
    size_mb = os.path.getsize(os.path.join(KG_OUTPUT_PATH, f)) / (1024*1024)
    print(f"  - {f}: {size_mb:.1f} MB")

Existing USDA triple files: 5
  - triples_part_1.csv: 233.1 MB
  - triples_part_2.csv: 239.7 MB
  - triples_part_3.csv: 239.9 MB
  - triples_part_4.csv: 244.3 MB
  - triples_part_5.csv: 29.5 MB


In [4]:
if REGENERATE_USDA or len(existing_usda_triples) == 0:
    print("Loading USDA cleaned data...")
    usda_file = os.path.join(USDA_DATA_PATH, 'cleaned_usda_data.csv')
    usda_df = pd.read_csv(usda_file, low_memory=False)
    print(f"Loaded {len(usda_df)} food records")
    print(f"Columns: {usda_df.columns.tolist()}")
else:
    print("Skipping USDA data loading - triples already exist")
    print("Set REGENERATE_USDA = True to regenerate")

Skipping USDA data loading - triples already exist
Set REGENERATE_USDA = True to regenerate


In [5]:
if REGENERATE_USDA or len(existing_usda_triples) == 0:
    print("Creating USDA triples...")
    triples = []
    count = 0
    
    for i, row in usda_df.iterrows():
        count += 1
        food_id = "food_" + str(row["fdc_id"])
        
        # Food name triple
        if pd.notna(row.get("description")):
            triples.append([food_id, "has_name", str(row["description"])])
        
        # Brand triple
        if pd.notna(row.get("brand_owner")):
            triples.append([food_id, "has_brand", str(row["brand_owner"])])
        
        # Ingredient triples - handle both string list and actual list
        ing_col = "ingredient_list" if "ingredient_list" in usda_df.columns else "ingredients_cleaned"
        if ing_col in usda_df.columns and pd.notna(row.get(ing_col)):
            try:
                ingredients = row[ing_col]
                if isinstance(ingredients, str):
                    if ingredients.startswith('['):
                        ingredients = eval(ingredients)  # Convert string list to list
                    else:
                        ingredients = [ing.strip() for ing in ingredients.split(',') if ing.strip()]
                
                for ing in ingredients:
                    if ing and len(str(ing).strip()) > 0:
                        triples.append([food_id, "contains_ingredient", str(ing).strip()])
            except:
                pass
        
        if count % 50000 == 0:
            print(f"Processed {count}/{len(usda_df)} foods... ({len(triples)} triples so far)")
    
    print(f"\nTotal USDA triples created: {len(triples)}")
else:
    print("Skipping USDA triple creation - already exists")

Skipping USDA triple creation - already exists


In [6]:
if REGENERATE_USDA or len(existing_usda_triples) == 0:
    # Convert to DataFrame and save in chunks
    all_triples = pd.DataFrame(triples, columns=["Subject", "Predicate", "Object"])
    
    # Save in 5M row chunks
    CHUNK_SIZE = 5000000
    part = 1
    for i in range(0, len(all_triples), CHUNK_SIZE):
        chunk = all_triples[i:i+CHUNK_SIZE]
        output_file = os.path.join(KG_OUTPUT_PATH, f"triples_part_{part}.csv")
        chunk.to_csv(output_file, index=False)
        print(f"Saved {output_file} ({len(chunk)} triples)")
        part += 1
    
    print(f"\nUSDA triples saved! Total: {len(all_triples)}")
else:
    print("USDA triples already exist - skipped")

USDA triples already exist - skipped


## Part 2: PubMed Knowledge Graph Triples

Create Ingredient → Disease triples with evidence metadata

In [7]:
# Load PubMed data
pubmed_file = os.path.join(PUBMED_DATA_PATH, 'Preprocessed_pubmed_data.xls')
print(f"Loading PubMed data from {pubmed_file}...")

# The file is actually CSV format
pubmed_df = pd.read_csv(pubmed_file, sep=',', on_bad_lines='warn', low_memory=False)
print(f"Loaded {len(pubmed_df)} PubMed records")
print(f"Columns: {pubmed_df.columns.tolist()}")
pubmed_df.head()

Loading PubMed data from ../data/pre-processed/pubmed/Preprocessed_pubmed_data.xls...
Loaded 30567 PubMed records
Columns: ['ingredient', 'pmid', 'title', 'year', 'journal', 'diseases_mentioned', 'mesh_disease_terms', 'relationship_type', 'key_findings', 'authors', 'pubmed_url']


,ingredient,pmid,title,year,journal,diseases_mentioned,mesh_disease_terms,relationship_type,key_findings,authors,pubmed_url
0,protein,41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology,"diabetes, insulin resistance, cardiovascular d...",NaN,Risk Factor,The results indicated that elevated TyG-ABSI v...,"Xinyi Shao, Zhaofu Tan, Lifu Sun",https://pubmed.ncbi.nlm.nih.gov/41390803/
1,protein,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports,alzheimer,NaN,Risk Factor,We analyzed electronic health records from UCL...,"Mingzhou Fu, Thai Tran, Bogdan Pasaniuc",https://pubmed.ncbi.nlm.nih.gov/41390778/
2,protein,41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports,"diabetes, obesity, diabetes mellitus",NaN,Risk Factor,"Of note, proximity ligation assay (PLA) reveal...","Ruofan Wang, Jianying Liu, Qian Li",https://pubmed.ncbi.nlm.nih.gov/41390701/
3,protein,41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports,"diabetes, type 2 diabetes",NaN,Associated/Neutral,See full abstract for details,"Sutichot D Nimkulrat, Matthew R Wagner, Bayley...",https://pubmed.ncbi.nlm.nih.gov/41390575/
4,protein,41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics,"inflammation, cardiovascular disease, stroke, ...",NaN,Associated/Neutral,The 5-lipoxygenase-activating protein (FLAP) i...,"Katharina Rataj, Ulrike Garscha",https://pubmed.ncbi.nlm.nih.gov/41390442/


In [8]:
# Create PubMed triples with evidence
print("Creating PubMed ingredient-disease triples...")

pubmed_triples = []

# Identify relevant columns
ingredient_col = 'ingredient' if 'ingredient' in pubmed_df.columns else pubmed_df.columns[0]
disease_col = None
for col in pubmed_df.columns:
    if 'disease' in col.lower():
        disease_col = col
        break

print(f"Using columns: ingredient='{ingredient_col}', disease='{disease_col}'")

for i, row in pubmed_df.iterrows():
    ingredient = str(row.get(ingredient_col, '')).strip()
    
    # Get disease - might be in different columns
    disease = ''
    if disease_col and pd.notna(row.get(disease_col)):
        disease = str(row[disease_col]).strip()
    
    if ingredient and disease and ingredient.lower() not in ['nan', ''] and disease.lower() not in ['nan', '']:
        triple = {
            'subject': ingredient,
            'predicate': 'RELATES_TO',
            'object': disease,
            'pmid': row.get('pmid', ''),
            'title': row.get('title', ''),
            'year': row.get('year', ''),
            'journal': row.get('journal', '')
        }
        pubmed_triples.append(triple)
    
    if (i + 1) % 10000 == 0:
        print(f"Processed {i+1}/{len(pubmed_df)} records...")

print(f"\nTotal PubMed triples created: {len(pubmed_triples)}")

Creating PubMed ingredient-disease triples...
Using columns: ingredient='ingredient', disease='diseases_mentioned'
Processed 10000/30567 records...
Processed 20000/30567 records...
Processed 30000/30567 records...

Total PubMed triples created: 30567


In [9]:
# Save PubMed triples as CSV
pubmed_triples_df = pd.DataFrame(pubmed_triples)
pubmed_output = os.path.join(KG_OUTPUT_PATH, 'pubmed_kg_triples.csv')
pubmed_triples_df.to_csv(pubmed_output, index=False)
print(f"Saved PubMed triples to {pubmed_output}")
print(f"Shape: {pubmed_triples_df.shape}")
pubmed_triples_df.head()

Saved PubMed triples to ../data/pre-processed/kg/pubmed_kg_triples.csv
Shape: (30567, 7)


,subject,predicate,object,pmid,title,year,journal
0,protein,RELATES_TO,"diabetes, insulin resistance, cardiovascular d...",41390803,Association between triglyceride-glucose index...,2025,Cardiovascular diabetology
1,protein,RELATES_TO,alzheimer,41390778,Multi-task learning identifies shared genetic ...,2025,Scientific reports
2,protein,RELATES_TO,"diabetes, obesity, diabetes mellitus",41390701,Obesity concurrent with gestational diabetes m...,2025,Scientific reports
3,protein,RELATES_TO,"diabetes, type 2 diabetes",41390575,The RNA-binding protein CPEB1 marks healthy ad...,2025,Scientific reports
4,protein,RELATES_TO,"inflammation, cardiovascular disease, stroke, ...",41390442,Genetic insights into 5-LOX-activating protein...,2025,Human genomics


In [10]:
# Create GraphML for visualization
print("Creating GraphML file for PubMed knowledge graph...")

G = nx.DiGraph()

# Add nodes and edges (limit for GraphML file size)
MAX_EDGES = 50000  # Limit for manageable GraphML file
for i, row in pubmed_triples_df.head(MAX_EDGES).iterrows():
    ingredient = row['subject']
    disease = row['object']
    
    # Add ingredient node
    if ingredient not in G:
        G.add_node(ingredient, node_type='Ingredient')
    
    # Add disease node
    if disease not in G:
        G.add_node(disease, node_type='Disease')
    
    # Add edge with evidence
    G.add_edge(ingredient, disease, 
               relationship='RELATES_TO',
               pmid=str(row.get('pmid', '')),
               year=str(row.get('year', '')))

graphml_output = os.path.join(KG_OUTPUT_PATH, 'pubmed_kg.graphml')
nx.write_graphml(G, graphml_output)
print(f"Saved GraphML to {graphml_output}")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Creating GraphML file for PubMed knowledge graph...
Saved GraphML to ../data/pre-processed/kg/pubmed_kg.graphml
Graph: 2548 nodes, 14242 edges


In [11]:
# Summary
print("\n" + "="*50)
print("Step 4: Knowledge Graph Triples - COMPLETE")
print("="*50)

print("\nUSDA Triples:")
usda_files = [f for f in os.listdir(KG_OUTPUT_PATH) if f.startswith('triples_part_')]
for f in usda_files:
    size = os.path.getsize(os.path.join(KG_OUTPUT_PATH, f)) / (1024*1024)
    print(f"  - {f}: {size:.1f} MB")

print("\nPubMed Triples:")
print(f"  - pubmed_kg_triples.csv: {len(pubmed_triples_df)} triples")
print(f"  - pubmed_kg.graphml: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


Step 4: Knowledge Graph Triples - COMPLETE

USDA Triples:
  - triples_part_1.csv: 233.1 MB
  - triples_part_2.csv: 239.7 MB
  - triples_part_3.csv: 239.9 MB
  - triples_part_4.csv: 244.3 MB
  - triples_part_5.csv: 29.5 MB

PubMed Triples:
  - pubmed_kg_triples.csv: 30567 triples
  - pubmed_kg.graphml: 2548 nodes, 14242 edges
